# Lời Chúa — Render audio ngắn bằng Google Colab

Notebook này chỉ chạy OmniVoice TTS trên GPU, không mở API và không tạo tunnel. Model, giọng mẫu, transcript và MP3 được lưu trong `MyDrive/LoiChuaAudio`.

**Lần đầu:** chọn `Runtime → Change runtime type → T4 GPU`, sau đó chạy lần lượt tất cả cell. Cell 3 sẽ yêu cầu upload ba file giọng. **Các lần sau:** chỉ cần chạy lại từ đầu; dữ liệu trong Drive sẽ được dùng lại.

> Chỉ sử dụng giọng của chính bạn hoặc giọng mà bạn có quyền sử dụng.

In [ ]:
#@title 1. Kiểm tra GPU và kết nối Google Drive
import os, shutil, subprocess, sys
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise SystemExit("Chưa có GPU. Chọn Runtime → Change runtime type → T4 GPU rồi chạy lại.")
print("GPU:", torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/LoiChuaAudio')
VOICE_DIR = DRIVE_ROOT / 'voices'
OUTPUT_DIR = DRIVE_ROOT / 'outputs'
HF_HOME = DRIVE_ROOT / 'huggingface-cache'
for folder in (VOICE_DIR, OUTPUT_DIR, HF_HOME):
    folder.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(HF_HOME)
os.environ['HUGGINGFACE_HUB_CACHE'] = str(HF_HOME / 'hub')
print("Thư mục làm việc:", DRIVE_ROOT)

In [ ]:
#@title 2. Cài OmniVoice TTS tối giản (lần đầu có thể mất vài phút)
OMNIVOICE_VERSION = 'v0.4.0'
REPO_DIR = Path('/content/OmniVoice-Studio')

def run(command, cwd=None):
    print('$', command if isinstance(command, str) else ' '.join(map(str, command)))
    subprocess.run(command, cwd=cwd, shell=isinstance(command, str), check=True)

run('apt-get -qq update && apt-get -qq install -y ffmpeg libsndfile1')
if not (REPO_DIR / '.git').exists():
    run(['git', 'clone', '--depth', '1', '--branch', OMNIVOICE_VERSION,
         'https://github.com/debpalash/OmniVoice-Studio.git', str(REPO_DIR)])

# Chỉ cài dependency cần cho TTS; bỏ ASR, dubbing, web UI và database.
run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
     'transformers>=5.3.0', 'accelerate', 'pydub', 'soundfile', 'sentencepiece'])
run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', str(REPO_DIR)])

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
from omnivoice import OmniVoice
print('OmniVoice sẵn sàng:', OMNIVOICE_VERSION)

In [ ]:
#@title 3. Upload các giọng còn thiếu vào Drive (chỉ cần làm một lần)
from google.colab import files

VOICE_FILES = {
    'hao': VOICE_DIR / 'voice_hao.mp3',
    'giang': VOICE_DIR / 'voice_giang.mp3',
    'trieu_duong': VOICE_DIR / 'voice_trieu_duong.mp3',
}

for voice_name, target in VOICE_FILES.items():
    if target.exists() and target.stat().st_size > 0:
        print(f'✓ {voice_name}: {target.name}')
        continue
    print(f'Upload file mẫu cho giọng {voice_name}...')
    uploaded = files.upload()
    if not uploaded:
        raise SystemExit(f'Chưa upload giọng {voice_name}.')
    source_name, content = next(iter(uploaded.items()))
    target.write_bytes(content)
    print(f'Đã lưu {source_name} → {target}')

print('Đã có đủ ba giọng trong Google Drive.')

In [ ]:
#@title 4. Nhập transcript của giọng mẫu (khuyến nghị)
# Transcript là đúng câu đang được nói trong từng file mẫu. Có thể để trống,
# nhưng bản TTS tối giản không tự chạy Whisper để tránh tải thêm model nặng.
HAO_REF_TEXT = "Xin chào, mình là Thế Hào. Rất vui vì hôm nay chúng ta lại có dịp ngồi lại với nhau, trong không gian nhỏ bé nhưng đầy yêu thương." #@param {type:"string"}
GIANG_REF_TEXT = "Xin chào, mình là Giang. Cảm ơn bạn đã ghé nghe thử giọng đọc của mình. Chất giọng này phù hợp với podcast, video kiến thức, review và sách nói." #@param {type:"string"}
TRIEU_DUONG_REF_TEXT = "Người ta là hỏa đất, học ăn, học nói, học gói, học mở." #@param {type:"string"}

VOICE_TRANSCRIPTS = {
    'hao': HAO_REF_TEXT.strip(),
    'giang': GIANG_REF_TEXT.strip(),
    'trieu_duong': TRIEU_DUONG_REF_TEXT.strip(),
}
missing = [name for name, value in VOICE_TRANSCRIPTS.items() if not value]
if missing:
    print('⚠ Chưa có transcript:', ', '.join(missing))
    print('Hãy nghe file mẫu, nhập nguyên văn câu được đọc rồi chạy lại cell này.')
else:
    import json
    (DRIVE_ROOT / 'voice_transcripts.json').write_text(
        json.dumps(VOICE_TRANSCRIPTS, ensure_ascii=False, indent=2), encoding='utf-8')
    print('Đã lưu transcript vào Drive.')

In [ ]:
#@title 5. Tải model lên GPU (mỗi phiên chỉ chạy một lần)
import json, gc, re, tempfile
import torchaudio

transcript_file = DRIVE_ROOT / 'voice_transcripts.json'
if transcript_file.exists():
    saved_transcripts = json.loads(transcript_file.read_text(encoding='utf-8'))
    for key, value in saved_transcripts.items():
        if value and not VOICE_TRANSCRIPTS.get(key):
            VOICE_TRANSCRIPTS[key] = value
missing = [name for name, value in VOICE_TRANSCRIPTS.items() if not value]
if missing:
    raise SystemExit('Thiếu transcript cho: ' + ', '.join(missing) + '. Hoàn thành cell 4 trước.')

if 'MODEL' not in globals():
    print('Đang tải model ~2.4 GB. Lần đầu sẽ lâu hơn; cache được giữ trong Drive...')
    MODEL = OmniVoice.from_pretrained(
        'k2-fsa/OmniVoice', device_map='cuda', dtype=torch.float16, load_asr=False)
    MODEL.eval()
    SAMPLE_RATE = getattr(MODEL, 'sampling_rate', 24000)
    VOICE_PROMPTS = {}
else:
    print('Đang dùng lại model đã có trong GPU.')

for voice_name, voice_path in VOICE_FILES.items():
    if voice_name not in VOICE_PROMPTS:
        print('Mã hóa giọng:', voice_name)
        VOICE_PROMPTS[voice_name] = MODEL.create_voice_clone_prompt(
            str(voice_path), ref_text=VOICE_TRANSCRIPTS[voice_name])
print(f'Model sẵn sàng trên CUDA; sample rate = {SAMPLE_RATE} Hz.')

In [ ]:
# 6. Hàm render phụng vụ — chạy một lần mỗi phiên
import unicodedata
from IPython.display import Audio, display

def safe_slug(value):
    value = re.sub(r'[\.,:;()\\/*?"<>|]', '', value.strip())
    value = re.sub(r'\s*-\s*', '-', value)
    return re.sub(r'\s+', '_', value) or 'custom_audio'

def split_liturgy_text(text, paragraph_pause, sentence_pause, major_pause, medium_pause):
    items = []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', line) if s.strip()]
        for sentence_index, sentence in enumerate(sentences):
            clauses = [c.strip() for c in re.split(r'(?<=[,;:—])\s+', sentence) if c.strip()]
            for clause_index, clause in enumerate(clauses):
                last_clause = clause_index == len(clauses) - 1
                last_sentence = sentence_index == len(sentences) - 1
                if last_clause:
                    pause = paragraph_pause if last_sentence else sentence_pause
                else:
                    pause = major_pause if clause.endswith((';', ':')) else medium_pause
                items.append((clause, float(pause)))
    return items

def trim_silence(audio, threshold=0.005):
    mask = audio.abs().squeeze(0) > threshold
    if not bool(mask.any()):
        return audio
    indices = torch.where(mask)[0]
    return audio[:, indices[0]:indices[-1] + 1]

def render_short_audio(*, ref, intro, content, section, voice, section_label='',
                       paragraph_pause=0.60, sentence_pause=0.45,
                       major_pause=0.30, medium_pause=0.25, num_step=16, overwrite=False):
    if voice not in VOICE_PROMPTS:
        raise ValueError('Giọng không hợp lệ: ' + voice)
    prefix = 'gospel' if section == 'gospel' else section
    out_dir = OUTPUT_DIR / ('gospels' if section == 'gospel' else f'readings/{section}')
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f'{prefix}_{safe_slug(ref)}.mp3'
    if out_path.exists() and not overwrite:
        print('Dùng file đã có:', out_path)
        display(Audio(str(out_path)))
        return out_path

    full_text = '\n'.join(part.strip() for part in (section_label, intro, content) if part.strip())
    chunks = split_liturgy_text(full_text, paragraph_pause, sentence_pause, major_pause, medium_pause)
    if not chunks:
        raise ValueError('Nội dung trống.')
    print(f'Render {len(chunks)} cụm bằng giọng {voice} trên CUDA...')
    audio_parts = []
    with torch.inference_mode():
        for index, (chunk_text, pause_seconds) in enumerate(chunks, 1):
            prompt_text = chunk_text.rstrip(',;:—').strip() or chunk_text
            generated = MODEL.generate(
                text=prompt_text, language='vi', voice_clone_prompt=VOICE_PROMPTS[voice],
                num_step=int(num_step), postprocess_output=True)[0]
            if generated.ndim == 1:
                generated = generated.unsqueeze(0)
            audio_parts.append(trim_silence(generated.detach().cpu()))
            audio_parts.append(torch.zeros((1, int(SAMPLE_RATE * pause_seconds))))
            print(f'  [{index}/{len(chunks)}] {prompt_text[:65]}')

    audio = torch.cat(audio_parts, dim=-1).float()
    peak = float(audio.abs().max())
    if peak > 0:
        audio = audio / peak * 0.97
    with tempfile.TemporaryDirectory() as temp_dir:
        wav_path = Path(temp_dir) / 'render.wav'
        torchaudio.save(str(wav_path), audio, SAMPLE_RATE, encoding='PCM_S', bits_per_sample=16)
        subprocess.run(['ffmpeg', '-hide_banner', '-loglevel', 'error', '-y',
                        '-i', str(wav_path), '-codec:a', 'libmp3lame', '-b:a', '96k',
                        '-ac', '1', str(out_path)], check=True)
    print('✓ Đã lưu:', out_path)
    display(Audio(str(out_path)))
    return out_path

print('Hàm render đã sẵn sàng.')

In [ ]:
#@title 7. Nhập nội dung và render — chạy lại cell này cho mỗi audio
REF = "1 V 3,5.7-12" #@param {type:"string"}
SECTION = "r1" #@param ["r1", "r2", "gospel"]
VOICE = "hao" #@param ["hao", "giang", "trieu_duong"]
SECTION_LABEL = "Bài đọc một." #@param {type:"string"}
INTRO = "Bài trích sách các Vua quyển thứ nhất." #@param {type:"string"}
CONTENT = "Hồi ấy, Đức Chúa phán với vua Sa-lô-môn." #@param {type:"string"}
PARAGRAPH_PAUSE = 0.60 #@param {type:"number"}
SENTENCE_PAUSE = 0.45 #@param {type:"number"}
MAJOR_PAUSE = 0.30 #@param {type:"number"}
MEDIUM_PAUSE = 0.25 #@param {type:"number"}
NUM_STEP = 16 #@param {type:"integer"}
OVERWRITE = False #@param {type:"boolean"}
DOWNLOAD_AFTER_RENDER = False #@param {type:"boolean"}

RESULT = render_short_audio(
    ref=REF, intro=INTRO, content=CONTENT, section=SECTION, voice=VOICE,
    section_label='' if SECTION == 'gospel' else SECTION_LABEL,
    paragraph_pause=PARAGRAPH_PAUSE, sentence_pause=SENTENCE_PAUSE,
    major_pause=MAJOR_PAUSE, medium_pause=MEDIUM_PAUSE,
    num_step=NUM_STEP, overwrite=OVERWRITE)
if DOWNLOAD_AFTER_RENDER:
    files.download(str(RESULT))

## Sau khi hoàn tất

File nằm trong `MyDrive/LoiChuaAudio/outputs`. Chạy lại **cell 7** để render audio tiếp theo; không chạy lại cell tải model. Khi xong, chọn `Runtime → Disconnect and delete runtime` để trả GPU cho Colab.